# Full Study — Final Block: Controlled Full Fine-Tuning Sensitivity v1.1

This is the **last experimental block** of the frozen breast-ultrasound foundation-model study.

Completed evidence entering this block:

- Phase-0A: URFM > architecture-matched ImageNet-MAE in 4/4 external scenarios.
- Phase-0B: UltraFedFM replicated the direction in 4/4 scenarios.
- Block 1 RQ2: both ultrasound-specific FMs retained the prespecified external advantage at
  20%, 50%, and 100% of the **source-training labels**.
- Block 1 RQ3: both ultrasound-specific FMs passed the prespecified common-pooling stability rule,
  although effect sizes attenuated under mean-patch pooling.

No new model family or paper direction is introduced here.

---

# RQ4

> **Does task adaptation by controlled full fine-tuning preserve, strengthen, or erase the
external-transfer advantage associated with ultrasound-specific pretraining?**

Primary architecture-controlled models:

1. URFM ViT-B/16, 224×224
2. UltraFedFM ViT-B/16, 224×224
3. ImageNet-MAE ViT-B/16, 224×224

Pooling:
- CLS only, matching the primary Phase-0A/B protocol.

Label availability:
- 100% of the frozen source-training partition only.

Sources:
- BUS-BRA, patient-level
- BrEaST, case-level

External targets:
- reciprocal BUS-BRA / BrEaST
- BUSI image-level external cohort

---

# Controlled full-fine-tuning recipe

This block tests **one shared prespecified adaptation recipe**.
It is not a model-specific hyperparameter competition.

For all three ViT-B encoders:

- optimizer: AdamW
- backbone learning rate: `1e-5`
- new linear classifier learning rate: `1e-4`
- weight decay: `1e-2`
- fixed 30 epochs
- no scheduler
- horizontal flip probability 0.5 during source training
- class-weighted cross-entropy
- best checkpoint selected only by frozen source-validation AUROC
- no target tuning
- no learning-rate sweep
- no model-specific augmentation
- no layer-wise LR decay
- no early stopping of the training budget

The best source-validation checkpoint is evaluated externally after all 30 epochs have completed.

---

# BUS-BRA patient-level training

BUS-BRA contains multiple images for some patients.

To preserve the patient as the source statistical unit:

1. a minibatch contains **patients**, not independent images;
2. all available images for each sampled patient are loaded;
3. the encoder/classifier produces image logits;
4. logits are averaged within each patient;
5. cross-entropy is applied to the patient-level mean logits.

Thus every patient contributes one loss term per epoch regardless of image count.

BrEaST has one lesion image per case and therefore reduces to ordinary case-level training.

Validation and external BUS-BRA scoring also average malignant margins within patient.

---

# Comparison to the completed frozen linear probe

The notebook does **not** retrain the linear probes.

It reuses:
- completed CLS feature caches;
- completed Phase-0A/B linear-head checkpoints.

For every model/source/seed/target it recomputes linear-probe target scores so that full-fine-tuning
and linear-probe effects can be compared on the same target units.

---

# Primary RQ4 quantities

## A. Full-FT ultrasound-FM advantage

For each USFM and external scenario:

\[
\Delta^{FT}_{FM-MAE}
=
AUC^{FT}_{FM}-AUC^{FT}_{MAE}
\]

seed-balanced across seeds 42/43/44.

## B. Within-model adaptation effect

\[
\Delta_{FT-LP,m}
=
AUC^{FT}_{m}-AUC^{LP}_{m}
\]

This asks whether full fine-tuning itself improves or harms external performance.

## C. Change in the ultrasound-pretraining advantage after adaptation

\[
\Delta_{adapt}
=
\Delta^{FT}_{FM-MAE}
-
\Delta^{LP}_{FM-MAE}
\]

This is the key adaptation-sensitivity quantity.

All use 3000 pathology-stratified paired target bootstraps.

---

# Frozen interpretation

For each USFM, the external advantage is called **preserved after full fine-tuning** only if all:

1. FT FM−MAE point effect > 0 in at least 3/4 scenarios;
2. median FT FM−MAE effect >= +0.02 AUROC;
3. paired-bootstrap `P(FM > MAE) >= .90` in at least 3/4 scenarios.

Adaptation change is summarized as:

- `STRENGTHENED` if >=3/4 scenario changes are positive and median change >= +0.02;
- `ATTENUATED` if >=3/4 scenario changes are negative and median change <= -0.02;
- otherwise `MIXED_OR_STABLE`.

These labels describe this experiment only.

---

# Final no-rescue rule

After this block completes:

- no third ultrasound FM;
- no LoRA;
- no VPT;
- no wavelet prompting;
- no extra readout family;
- no extra pooling rule;
- no target tuning;
- no post-hoc source/target exclusion;
- no extra seed to repair an unfavorable result.

Regardless of whether full fine-tuning preserves or erases the advantage, the experimental phase is
finished and the next step is manuscript construction with the claim narrowed to the observed
evidence.


### v1.1 history-audit recovery patch

A completed run may have a truncated per-epoch history CSV after Drive/runtime synchronization,
even when its completion status and saved external scores are intact.

v1.1 therefore treats a run as scientifically complete when:
- `complete_30_epochs == true`,
- `external_scores_saved == true`,
- source-validation scores exist,
- external target scores exist.

The history CSV remains audited, but an incomplete history artifact is recorded explicitly rather
than causing the whole analysis to fail. Missing epoch rows are never reconstructed or invented.

No model, seed, training recipe, checkpoint-selection rule, external score, bootstrap, or scientific
interpretation changes.


In [ ]:
!pip -q install timm==1.0.28 huggingface_hub kagglehub pandas scikit-learn pillow tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, re, gc, json, random, zipfile, tempfile, shutil
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
import kagglehub
from huggingface_hub import hf_hub_download, login
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm

ROOT = Path('/content/drive/MyDrive/SPATIOSPECTRAL_PEFT_PILOT')

PHASEA = ROOT/'fm_external_transfer_phase0_v1_0'
PHASEA_OUT = PHASEA/'outputs'
PHASEA_PROG = PHASEA/'progress'
PHASEA_CKPT = PHASEA/'checkpoints'

PHASEB = ROOT/'ultrafedfm_phase0b_v1_0'
PHASEB_OUT = PHASEB/'outputs'
PHASEB_PROG = PHASEB/'progress'
PHASEB_CKPT = PHASEB/'checkpoints'

PACKA = ROOT/'reviewer_proof_pack_A_v1_0'
PACKAO = PACKA/'outputs'
D1 = ROOT/'linear_readout_control_v1_0'
D1O = D1/'outputs'

URFM_REPO_ID = 'QingboKang/URFM'
URFM_FILENAME = 'mae_vit_base_patch16_dec768d8b_all_biomedclip_1199.pth'
URFM_LOCAL = ROOT/'external_models'/'URFM'/URFM_FILENAME

ULTRAFED_LOCAL = (
    ROOT/'external_models'/'UltraFedFM'/
    'official_pretrained_ultrafedfm_checkpoint.pth'
)

EXP = ROOT/'fullstudy_final_fullft_v1_0'
OUT = EXP/'outputs'
RUNS = EXP/'runs'
for p in [EXP,OUT,RUNS]:
    p.mkdir(parents=True,exist_ok=True)

MODELS = ['URFM','UltraFedFM','MAE']
USFMS = ['URFM','UltraFedFM']
SOURCES = ['BUSBRA','BREAST']
SEEDS = [42,43,44]

EPOCHS = 30
PATIENT_BATCH = 8
BACKBONE_LR = 1e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = 1e-2
HFLIP_P = 0.5

INFER_BATCH = 32

BOOTSTRAP_REPS = 3000
BOOTSTRAP_SEED = 101001

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:",DEVICE)
if DEVICE.type=='cuda':
    print(torch.cuda.get_device_name(0))

if DEVICE.type!='cuda':
    raise RuntimeError("GPU is required for controlled full fine-tuning.")

CONFIG = {
    'experiment':'Full Study Final Block: controlled full fine-tuning v1.1',
    'models':MODELS,
    'sources':SOURCES,
    'seeds':SEEDS,
    'epochs':EPOCHS,
    'patient_batch_size':PATIENT_BATCH,
    'backbone_lr':BACKBONE_LR,
    'head_lr':HEAD_LR,
    'weight_decay':WEIGHT_DECAY,
    'scheduler':None,
    'train_augmentation':{'horizontal_flip_p':HFLIP_P},
    'pooling':'CLS',
    'BUS_BRA_training_unit':'patient; all patient images averaged at logit level before CE',
    'BREAST_training_unit':'case',
    'best_checkpoint':'highest source-validation AUROC after fixed 30-epoch training budget',
    'target_tuning':False,
    'bootstrap_reps':BOOTSTRAP_REPS,
    'post_block':'WRITE_MANUSCRIPT_NO_RESCUE_EXPERIMENTS'
}
(EXP/'FROZEN_FULLFT_CONFIG.json').write_text(
    json.dumps(CONFIG,indent=2),encoding='utf-8'
)


## 1. Utilities and manifests

In [ ]:
IMAGENET_MEAN=torch.tensor([0.485,0.456,0.406],dtype=torch.float32).view(3,1,1)
IMAGENET_STD=torch.tensor([0.229,0.224,0.225],dtype=torch.float32).view(3,1,1)

def seed_everything(seed):
    os.environ['PYTHONHASHSEED']=str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

def atomic_csv(df,path):
    path=Path(path)
    tmp=path.with_suffix(path.suffix+'.tmp')
    df.to_csv(tmp,index=False)
    os.replace(tmp,path)

def atomic_json(obj,path):
    path=Path(path)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2),encoding='utf-8')
    os.replace(tmp,path)

def canonical_patient_key(v):
    s=str(v).strip()
    if re.fullmatch(r'\d+(?:\.0+)?',s):
        return str(int(float(s)))
    return s.lower()

def all_exist(values):
    return bool(np.all([Path(str(x)).exists() for x in values]))

def transform_pil(img,flip=False):
    img=img.convert('RGB').resize((224,224),Image.Resampling.BILINEAR)
    if flip:
        img=img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    x=torch.from_numpy(np.asarray(img,dtype=np.float32)/255.0).permute(2,0,1)
    return (x-IMAGENET_MEAN)/IMAGENET_STD

def auc_ap(y,s):
    return float(roc_auc_score(y,s)),float(average_precision_score(y,s))

def class_weights(y):
    c=np.bincount(np.asarray(y,dtype=int),minlength=2)
    w=c.sum()/(2*np.maximum(c,1))
    w=w/w.mean()
    return torch.tensor(w,dtype=torch.float32,device=DEVICE)

# ---------- manifests ----------
busi_candidates=[
    PACKAO/'BUSI_RUNTIME_REMAPPED_MANIFEST.csv',
    D1O/'BUSI_LINEAR_CONTROL_MANIFEST.csv',
]
busi_path=next((p for p in busi_candidates if p.exists()),None)

busbra_candidates=[
    PACKAO/'BUSBRA_RUNTIME_REMAPPED_MANIFEST.csv',
    D1O/'BUSBRA_EXTERNAL_MANIFEST.csv',
]
busbra_path=next((p for p in busbra_candidates if p.exists()),None)

breast_path=PACKAO/'BREAST_LESION_MANIFEST.csv'

if busi_path is None or busbra_path is None or not breast_path.exists():
    raise FileNotFoundError("Required existing manifests are missing.")

busi=pd.read_csv(busi_path)
busbra=pd.read_csv(busbra_path)
breast=pd.read_csv(breast_path)

if not all_exist(busi.image_path):
    print("Remapping stale BUSI paths with KaggleHub.")
    root=Path(kagglehub.dataset_download('aryashah2k/breast-ultrasound-images-dataset'))
    idx={}
    for p in root.rglob('*.png'):
        if '_mask' in p.stem.lower():
            continue
        idx[(p.parent.name.strip().lower(),p.stem)] = str(p)

    resolved=[]
    missing=[]
    for _,r in busi.iterrows():
        key=(str(r.label_text).strip().lower(),str(r.image_id).strip())
        pp=idx.get(key)
        if pp is None: missing.append(key)
        resolved.append(pp)
    if missing:
        raise RuntimeError(f"BUSI remap failed: {missing[:10]}")
    busi=busi.copy()
    busi['image_path']=resolved

if not all_exist(busbra.image_path):
    candidates=[
        ROOT/'data'/'BUSBRA_OFFICIAL'/'BUSBRA'/'Images',
        ROOT/'data'/'BUS-BRA'/'Images',
    ]
    image_dir=next((p for p in candidates if p.exists()),None)
    if image_dir is None:
        raise FileNotFoundError("BUS-BRA raw image directory not found.")
    remapped=[str(image_dir/f'{str(iid)}.png') for iid in busbra.image_id]
    if not all_exist(remapped):
        raise RuntimeError("BUS-BRA path remap failed.")
    busbra=busbra.copy()
    busbra['image_path']=remapped

if not all_exist(breast.image_path):
    raise FileNotFoundError("BrEaST image paths are stale.")

busi_all=pd.concat([
    busi[busi.split=='train'].reset_index(drop=True),
    busi[busi.split=='val'].reset_index(drop=True),
],ignore_index=True)

busbra['patient_key']=busbra.patient_key.map(canonical_patient_key)
breast['patient_key']=breast.patient_key.astype(str)

assert len(busi_all)==647
assert len(busbra)==1875 and busbra.patient_key.nunique()==1064
assert len(breast)==252 and breast.patient_key.nunique()==252

print("PASS manifests.")


## 2. Recover exact frozen source train/validation units

In [ ]:
split_paths={
    'BUSBRA':PHASEA_OUT/'BUSBRA_PHASE0_SOURCE_SPLIT.csv',
    'BREAST':PHASEA_OUT/'BREAST_PHASE0_SOURCE_SPLIT.csv',
}
saved_splits={k:pd.read_csv(v) for k,v in split_paths.items()}

def make_patient_table(source):
    if source=='BUSBRA':
        rows=[]
        for (pid,y),g in busbra.groupby(['patient_key','y'],sort=True):
            rows.append({
                'patient_key':str(pid),
                'y':int(y),
                'image_paths':list(g.image_path.astype(str))
            })
        return pd.DataFrame(rows)

    if source=='BREAST':
        return pd.DataFrame({
            'patient_key':breast.patient_key.astype(str),
            'y':breast.y.astype(int),
            'image_paths':[[str(p)] for p in breast.image_path]
        }).reset_index(drop=True)

    raise ValueError(source)

patient_tables={s:make_patient_table(s) for s in SOURCES}
source_indices={}

for source in SOURCES:
    pt=patient_tables[source]
    saved=saved_splits[source].copy()
    saved['patient_key']=saved.patient_key.astype(str)

    smap=dict(zip(saved.patient_key,saved.source_split))
    role=pt.patient_key.astype(str).map(smap)
    if role.isna().any():
        raise RuntimeError(f"{source}: frozen split could not be recovered.")

    tr=np.flatnonzero(role.to_numpy()=='train')
    va=np.flatnonzero(role.to_numpy()=='val')
    source_indices[source]=(tr,va)

    assert set(pt.iloc[tr].patient_key).isdisjoint(set(pt.iloc[va].patient_key))

    print(
        source,
        "train patients/cases",len(tr),
        "val",len(va),
        "train class counts",np.bincount(pt.iloc[tr].y.to_numpy(int),minlength=2).tolist()
    )


## 3. Patient-batch dataset

A BUS-BRA training item is one patient and contains all available images for that patient.

The custom collate concatenates images while preserving a patient-owner index.
Image logits are averaged back to one patient logit before cross-entropy.


In [ ]:
class PatientTrainDataset(Dataset):
    def __init__(self,patient_df,indices,seed):
        self.df=patient_df.iloc[np.asarray(indices,dtype=int)].reset_index(drop=True)
        self.seed=int(seed)
        self.epoch=0

    def set_epoch(self,epoch):
        self.epoch=int(epoch)

    def __len__(self):
        return len(self.df)

    def __getitem__(self,i):
        r=self.df.iloc[i]
        xs=[]
        for j,p in enumerate(r.image_paths):
            # Deterministic augmentation per epoch / patient / image / run seed.
            aug_seed=(
                self.seed*10000019
                + self.epoch*100003
                + i*1009
                + j*97
            )
            rng=np.random.default_rng(aug_seed)
            flip=bool(rng.random()<HFLIP_P)
            img=Image.open(p).convert('RGB')
            xs.append(transform_pil(img,flip=flip))

        return {
            'images':torch.stack(xs),
            'y':int(r.y),
            'patient_key':str(r.patient_key)
        }

def patient_collate(batch):
    images=[]
    owners=[]
    ys=[]
    keys=[]

    for bi,item in enumerate(batch):
        n=item['images'].shape[0]
        images.append(item['images'])
        owners.extend([bi]*n)
        ys.append(item['y'])
        keys.append(item['patient_key'])

    return {
        'images':torch.cat(images,dim=0),
        'owners':torch.tensor(owners,dtype=torch.long),
        'y':torch.tensor(ys,dtype=torch.long),
        'patient_key':keys
    }

class ImageInferDataset(Dataset):
    def __init__(self,df):
        self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        img=Image.open(r.image_path).convert('RGB')
        return {
            'x':transform_pil(img,flip=False),
            'idx':i
        }


## 4. Pretrained encoder builders

In [ ]:
def filtered_load_vit(checkpoint_path):
    ck=torch.load(checkpoint_path,map_location='cpu',weights_only=False)
    if not isinstance(ck,dict):
        raise RuntimeError("Checkpoint is not a dict.")

    if 'model' in ck and isinstance(ck['model'],dict):
        state=ck['model']
    elif 'state_dict' in ck and isinstance(ck['state_dict'],dict):
        state=ck['state_dict']
    else:
        state=ck

    cleaned={}
    for k,v in state.items():
        nk=k
        for prefix in ['module.','model.','encoder.']:
            if nk.startswith(prefix):
                nk=nk[len(prefix):]
        cleaned[nk]=v

    model=timm.create_model(
        'vit_base_patch16_224',
        pretrained=False,
        num_classes=0,
        global_pool='token'
    )

    target=model.state_dict()
    filt={}
    for k,v in cleaned.items():
        if k.startswith('decoder') or k=='mask_token' or k.startswith('head.'):
            continue
        if k in target and tuple(v.shape)==tuple(target[k].shape):
            filt[k]=v

    msg=model.load_state_dict(filt,strict=False)

    critical=[
        'patch_embed.proj.weight','cls_token','pos_embed',
        'blocks.0.attn.qkv.weight','blocks.11.attn.qkv.weight',
        'norm.weight','norm.bias'
    ]
    absent=[k for k in critical if k not in filt]
    if absent:
        raise RuntimeError(f"Critical encoder keys missing: {absent}")

    bad_missing=[k for k in msg.missing_keys if not k.startswith('head.')]
    if bad_missing:
        raise RuntimeError(f"Unexpected missing encoder keys: {bad_missing[:20]}")

    return model

def resolve_urfm():
    if URFM_LOCAL.exists():
        return str(URFM_LOCAL)

    print("URFM checkpoint absent; gated Hugging Face login required.")
    login()
    p=hf_hub_download(repo_id=URFM_REPO_ID,filename=URFM_FILENAME)
    URFM_LOCAL.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(p,URFM_LOCAL)
    return str(URFM_LOCAL)

def build_backbone(model_name):
    if model_name=='URFM':
        model=filtered_load_vit(resolve_urfm())

    elif model_name=='UltraFedFM':
        if not ULTRAFED_LOCAL.exists():
            raise FileNotFoundError(ULTRAFED_LOCAL)
        model=filtered_load_vit(str(ULTRAFED_LOCAL))

    elif model_name=='MAE':
        try:
            model=timm.create_model(
                'vit_base_patch16_224.mae',
                pretrained=True,
                num_classes=0,
                global_pool='token'
            )
        except Exception:
            model=timm.create_model(
                'hf_hub:timm/vit_base_patch16_224.mae',
                pretrained=True,
                num_classes=0,
                global_pool='token'
            )
    else:
        raise ValueError(model_name)

    with torch.no_grad():
        z=model(torch.zeros(1,3,224,224))
    if tuple(z.shape)!=(1,768):
        raise RuntimeError(f"{model_name}: unexpected output shape {tuple(z.shape)}")

    return model


## 5. Inference helpers

In [ ]:
@torch.no_grad()
def score_image_manifest(backbone,head,df):
    ds=ImageInferDataset(df)
    loader=DataLoader(
        ds,batch_size=INFER_BATCH,shuffle=False,num_workers=0,pin_memory=True
    )

    backbone.eval()
    head.eval()
    scores=np.empty(len(df),dtype=np.float64)

    for b in loader:
        x=b['x'].to(DEVICE,non_blocking=True)
        z=backbone(x)
        logits=head(z)
        s=(logits[:,1]-logits[:,0]).detach().cpu().numpy()
        scores[b['idx'].numpy().astype(int)]=s

    return scores

def aggregate_busbra_patient_scores(img_scores):
    t=busbra[['patient_key','y']].copy().reset_index(drop=True)
    t['score']=np.asarray(img_scores,dtype=float)
    return (
        t.groupby(['patient_key','y'],as_index=False)
        .agg(score=('score','mean'))
        .sort_values('patient_key')
        .reset_index(drop=True)
    )

@torch.no_grad()
def score_source_val(backbone,head,source,val_indices):
    pt=patient_tables[source].iloc[np.asarray(val_indices,dtype=int)]
    rows=[]
    for _,r in pt.iterrows():
        for p in r.image_paths:
            rows.append({
                'patient_key':str(r.patient_key),
                'y':int(r.y),
                'image_path':str(p)
            })
    df=pd.DataFrame(rows)
    img_s=score_image_manifest(backbone,head,df)
    df['score']=img_s

    pat=(
        df.groupby(['patient_key','y'],as_index=False)
        .agg(score=('score','mean'))
        .sort_values('patient_key')
        .reset_index(drop=True)
    )
    auc,ap=auc_ap(pat.y,pat.score)
    return auc,ap,pat

@torch.no_grad()
def score_external_targets(backbone,head):
    out={}

    # BUSI image-level
    s=score_image_manifest(backbone,head,busi_all)
    out['BUSI']=pd.DataFrame({
        'unit_id':busi_all.image_id.astype(str),
        'y':busi_all.y.astype(int),
        'score':s
    })

    # BrEaST case-level
    s=score_image_manifest(backbone,head,breast)
    out['BREAST']=pd.DataFrame({
        'unit_id':breast.patient_key.astype(str),
        'y':breast.y.astype(int),
        'score':s
    })

    # BUS-BRA patient-level
    s=score_image_manifest(backbone,head,busbra)
    p=aggregate_busbra_patient_scores(s)
    out['BUSBRA']=pd.DataFrame({
        'unit_id':p.patient_key.astype(str),
        'y':p.y.astype(int),
        'score':p.score.astype(float)
    })

    return out


## 6. One interruption-safe full-fine-tuning run

Completed runs are skipped.

During an active run, the best checkpoint is kept in local runtime storage only.
After all 30 epochs:
- best source checkpoint is loaded;
- external target scores are saved permanently to Drive;
- training history/status are saved;
- the temporary full model checkpoint is deleted.

If Colab disconnects during one active run, only that unfinished run is repeated.
Previously completed runs are not repeated.


In [ ]:
def run_paths(model,source,seed):
    tag=f'{model}_{source}_seed{seed}'
    return {
        'history':RUNS/f'{tag}_history.csv',
        'status':RUNS/f'{tag}_status.json',
        'scores':RUNS/f'{tag}_external_scores.csv',
        'source_val':RUNS/f'{tag}_source_val_scores.csv'
    }

def run_complete(model,source,seed):
    p=run_paths(model,source,seed)
    if not all(x.exists() for x in p.values()):
        return False
    try:
        st=json.loads(p['status'].read_text(encoding='utf-8'))
        return bool(st.get('complete_30_epochs',False) and st.get('external_scores_saved',False))
    except Exception:
        return False

def fullft_run(model_name,source,seed):
    paths=run_paths(model_name,source,seed)

    if run_complete(model_name,source,seed):
        print("SKIP completed:",model_name,source,seed)
        return

    print("\\nRUN",model_name,source,"seed",seed)
    seed_everything(seed)

    backbone=build_backbone(model_name).to(DEVICE)
    head=nn.Linear(768,2).to(DEVICE)

    # All parameters trainable.
    for p in backbone.parameters():
        p.requires_grad=True
    for p in head.parameters():
        p.requires_grad=True

    opt=torch.optim.AdamW(
        [
            {'params':backbone.parameters(),'lr':BACKBONE_LR},
            {'params':head.parameters(),'lr':HEAD_LR},
        ],
        weight_decay=WEIGHT_DECAY
    )

    pt=patient_tables[source]
    tr,va=source_indices[source]

    train_ds=PatientTrainDataset(pt,tr,seed)
    cw=class_weights(pt.iloc[tr].y.to_numpy(int))

    tmpdir=Path('/content/fullft_tmp')
    tmpdir.mkdir(parents=True,exist_ok=True)
    best_path=tmpdir/f'best_{model_name}_{source}_{seed}.pt'
    if best_path.exists():
        best_path.unlink()

    best_auc=-np.inf
    best_epoch=None
    hist=[]

    for epoch in range(1,EPOCHS+1):
        train_ds.set_epoch(epoch)

        gen=torch.Generator()
        gen.manual_seed(seed*1000+epoch)

        loader=DataLoader(
            train_ds,
            batch_size=PATIENT_BATCH,
            shuffle=True,
            num_workers=0,
            pin_memory=True,
            collate_fn=patient_collate,
            generator=gen
        )

        backbone.train()
        head.train()
        losses=[]

        for b in tqdm(
            loader,
            desc=f'{model_name}-{source}-s{seed}-e{epoch}',
            leave=False
        ):
            x=b['images'].to(DEVICE,non_blocking=True)
            owners=b['owners'].to(DEVICE,non_blocking=True)
            yy=b['y'].to(DEVICE,non_blocking=True)

            opt.zero_grad(set_to_none=True)

            z=backbone(x)
            logits_img=head(z)

            n_pat=len(yy)
            logits_pat=torch.zeros(
                n_pat,2,device=DEVICE,dtype=logits_img.dtype
            )
            logits_pat.index_add_(0,owners,logits_img)

            counts=torch.bincount(
                owners,minlength=n_pat
            ).to(logits_img.dtype).unsqueeze(1)

            logits_pat=logits_pat/counts

            loss=F.cross_entropy(
                logits_pat,yy,weight=cw
            )
            loss.backward()
            opt.step()

            losses.append(float(loss.item()))

        val_auc,val_ap,_=score_source_val(
            backbone,head,source,va
        )

        hist.append({
            'model':model_name,
            'source':source,
            'seed':seed,
            'epoch':epoch,
            'train_loss':float(np.mean(losses)),
            'source_val_auc':val_auc,
            'source_val_auprc':val_ap
        })

        # Save history after every completed epoch (small file).
        atomic_csv(pd.DataFrame(hist),paths['history'])

        if val_auc>best_auc:
            best_auc=val_auc
            best_epoch=epoch

            # Local temporary checkpoint only.
            torch.save({
                'backbone':{
                    k:v.detach().cpu().clone()
                    for k,v in backbone.state_dict().items()
                },
                'head':{
                    k:v.detach().cpu().clone()
                    for k,v in head.state_dict().items()
                },
                'epoch':epoch,
                'source_val_auc':best_auc
            },best_path)

    if len(hist)!=EPOCHS:
        raise RuntimeError("Run did not complete the frozen 30 epochs.")

    ck=torch.load(best_path,map_location='cpu',weights_only=False)
    backbone.load_state_dict(ck['backbone'])
    head.load_state_dict(ck['head'])
    backbone=backbone.to(DEVICE).eval()
    head=head.to(DEVICE).eval()

    val_auc,val_ap,val_scores=score_source_val(
        backbone,head,source,va
    )
    atomic_csv(val_scores,paths['source_val'])

    ext=score_external_targets(backbone,head)
    score_rows=[]
    for target,df in ext.items():
        d=df.copy()
        d.insert(0,'target',target)
        score_rows.append(d)
    score_long=pd.concat(score_rows,ignore_index=True)
    atomic_csv(score_long,paths['scores'])

    st={
        'model':model_name,
        'source':source,
        'seed':seed,
        'complete_30_epochs':True,
        'best_epoch':int(best_epoch),
        'best_source_val_auc':float(best_auc),
        'reloaded_source_val_auc':float(val_auc),
        'reloaded_source_val_auprc':float(val_ap),
        'external_scores_saved':True,
        'patient_level_training':bool(source=='BUSBRA'),
        'backbone_lr':BACKBONE_LR,
        'head_lr':HEAD_LR,
        'weight_decay':WEIGHT_DECAY
    }
    atomic_json(st,paths['status'])

    if best_path.exists():
        best_path.unlink()

    del backbone,head,opt
    gc.collect()
    torch.cuda.empty_cache()

    print("COMPLETE:",model_name,source,seed,"best epoch",best_epoch,"val AUC",best_auc)


## 7. Execute all 18 full-FT runs

In [ ]:
for model in MODELS:
    for source in SOURCES:
        for seed in SEEDS:
            fullft_run(model,source,seed)

print("All requested runs completed or were safely reused.")


## 8. Audit and consolidate full-FT runs

In [ ]:
status_rows=[]
histories=[]
history_audit_rows=[]
ft_score_rows=[]

for model in MODELS:
    for source in SOURCES:
        for seed in SEEDS:
            p=run_paths(model,source,seed)

            # Completion authority: final status + saved source-val + saved external scores.
            if not p['status'].exists():
                raise RuntimeError(f"Missing status file: {model}/{source}/seed{seed}")
            if not p['source_val'].exists():
                raise RuntimeError(f"Missing source-val scores: {model}/{source}/seed{seed}")
            if not p['scores'].exists():
                raise RuntimeError(f"Missing external scores: {model}/{source}/seed{seed}")

            st=json.loads(p['status'].read_text(encoding='utf-8'))
            if not bool(st.get('complete_30_epochs',False)):
                raise RuntimeError(f"Run not marked 30-epoch complete: {model}/{source}/seed{seed}")
            if not bool(st.get('external_scores_saved',False)):
                raise RuntimeError(f"External scores not marked saved: {model}/{source}/seed{seed}")

            status_rows.append(st)

            # History is audited as an artifact, but is not required for downstream RQ4 metrics.
            if p['history'].exists():
                h=pd.read_csv(p['history']).copy()
                if len(h):
                    h['model']=model
                    h['source']=source
                    h['seed']=seed
                    histories.append(h)

                    epochs = sorted(pd.to_numeric(h['epoch'],errors='coerce').dropna().astype(int).unique().tolist())
                    missing_epochs = sorted(set(range(1,EPOCHS+1)) - set(epochs))
                    duplicate_epochs = sorted(
                        h.loc[h.duplicated('epoch',keep=False),'epoch']
                        .dropna().astype(int).unique().tolist()
                    )
                    max_epoch = int(max(epochs)) if epochs else None
                    complete_history = bool(
                        len(h)==EPOCHS
                        and epochs==list(range(1,EPOCHS+1))
                        and len(duplicate_epochs)==0
                    )
                else:
                    missing_epochs=list(range(1,EPOCHS+1))
                    duplicate_epochs=[]
                    max_epoch=None
                    complete_history=False
            else:
                h=None
                missing_epochs=list(range(1,EPOCHS+1))
                duplicate_epochs=[]
                max_epoch=None
                complete_history=False

            history_audit_rows.append({
                'model':model,
                'source':source,
                'seed':seed,
                'status_complete_30_epochs':bool(st.get('complete_30_epochs',False)),
                'external_scores_saved':bool(st.get('external_scores_saved',False)),
                'history_file_exists':bool(p['history'].exists()),
                'history_rows':0 if h is None else int(len(h)),
                'history_max_epoch':max_epoch,
                'missing_epochs':';'.join(map(str,missing_epochs)),
                'duplicate_epochs':';'.join(map(str,duplicate_epochs)),
                'history_artifact_complete':complete_history,
                'history_audit_label':(
                    'COMPLETE_HISTORY'
                    if complete_history
                    else 'INCOMPLETE_HISTORY_ARTIFACT'
                )
            })

            s=pd.read_csv(p['scores'])
            if s.empty:
                raise RuntimeError(f"Empty external score file: {model}/{source}/seed{seed}")

            # Minimal target presence audit.
            observed_targets=set(s['target'].astype(str).unique().tolist())
            expected_targets={'BUSI','BREAST','BUSBRA'}
            if observed_targets != expected_targets:
                raise RuntimeError(
                    f"External target set mismatch {model}/{source}/seed{seed}: "
                    f"{sorted(observed_targets)}"
                )

            s.insert(0,'seed',seed)
            s.insert(0,'source',source)
            s.insert(0,'model',model)
            ft_score_rows.append(s)

status=pd.DataFrame(status_rows)
history_audit=pd.DataFrame(history_audit_rows)
hist=pd.concat(histories,ignore_index=True) if histories else pd.DataFrame()
ft_scores=pd.concat(ft_score_rows,ignore_index=True)

assert len(status)==18
assert bool(status.complete_30_epochs.all())
assert bool(status.external_scores_saved.all())
assert len(ft_score_rows)==18

atomic_csv(status,OUT/'FULLFT_TRAINING_STATUS.csv')
atomic_csv(history_audit,OUT/'FULLFT_HISTORY_ARTIFACT_AUDIT.csv')
atomic_csv(hist,OUT/'FULLFT_ALL_HISTORIES.csv')
atomic_csv(ft_scores,OUT/'FULLFT_EXTERNAL_TARGET_SCORES.csv')

print("PASS: 18/18 runs have complete status and saved external scores.")
n_incomplete=int((~history_audit.history_artifact_complete).sum())
print("Incomplete history artifacts:",n_incomplete)

if n_incomplete:
    display(
        history_audit.loc[
            ~history_audit.history_artifact_complete,
            ['model','source','seed','history_rows','history_max_epoch',
             'missing_epochs','history_audit_label']
        ]
    )

display(status.round(4))


## 9. Recompute the completed linear-probe target scores

No linear-probe training is repeated.


In [ ]:
cls_paths={
    ('URFM','BUSI'):PHASEA_PROG/'URFM_224_BUSI_features.npy',
    ('URFM','BUSBRA'):PHASEA_PROG/'URFM_224_BUSBRA_features.npy',
    ('URFM','BREAST'):PHASEA_PROG/'URFM_224_BREAST_features.npy',

    ('MAE','BUSI'):PHASEA_PROG/'ImageNet_MAE_224_BUSI_features.npy',
    ('MAE','BUSBRA'):PHASEA_PROG/'ImageNet_MAE_224_BUSBRA_features.npy',
    ('MAE','BREAST'):PHASEA_PROG/'ImageNet_MAE_224_BREAST_features.npy',

    ('UltraFedFM','BUSI'):PHASEB_PROG/'UltraFedFM_224_BUSI_features.npy',
    ('UltraFedFM','BUSBRA'):PHASEB_PROG/'UltraFedFM_224_BUSBRA_features.npy',
    ('UltraFedFM','BREAST'):PHASEB_PROG/'UltraFedFM_224_BREAST_features.npy',
}

features={
    k:np.asarray(np.load(v,mmap_mode='r'),dtype=np.float32)
    for k,v in cls_paths.items()
}

def load_lp_head(model,source,seed):
    if model=='URFM':
        p=PHASEA_CKPT/f'URFM_224_{source}_seed{seed}_best.pt'
    elif model=='UltraFedFM':
        p=PHASEB_CKPT/f'UltraFedFM_224_{source}_seed{seed}_best.pt'
    elif model=='MAE':
        p=PHASEA_CKPT/f'ImageNet_MAE_224_{source}_seed{seed}_best.pt'
    else:
        raise ValueError(model)

    if not p.exists():
        raise FileNotFoundError(p)

    ck=torch.load(p,map_location='cpu',weights_only=False)
    h=nn.Linear(768,2)
    h.load_state_dict(ck['state_dict'])
    h.eval()
    return h

@torch.no_grad()
def score_features(head,X):
    X=np.asarray(X,dtype=np.float32)
    logits=head(torch.from_numpy(X))
    return (logits[:,1]-logits[:,0]).numpy().astype(float)

lp_rows=[]

for model in MODELS:
    for source in SOURCES:
        for seed in SEEDS:
            h=load_lp_head(model,source,seed)

            # BUSI
            s=score_features(h,features[(model,'BUSI')])
            lp_rows.append(pd.DataFrame({
                'model':model,'source':source,'seed':seed,'target':'BUSI',
                'unit_id':busi_all.image_id.astype(str),
                'y':busi_all.y.astype(int),'score':s
            }))

            # BREAST
            s=score_features(h,features[(model,'BREAST')])
            lp_rows.append(pd.DataFrame({
                'model':model,'source':source,'seed':seed,'target':'BREAST',
                'unit_id':breast.patient_key.astype(str),
                'y':breast.y.astype(int),'score':s
            }))

            # BUSBRA patient
            s=score_features(h,features[(model,'BUSBRA')])
            t=busbra[['patient_key','y']].copy()
            t['score']=s
            p=(
                t.groupby(['patient_key','y'],as_index=False)
                .agg(score=('score','mean'))
                .sort_values('patient_key').reset_index(drop=True)
            )
            lp_rows.append(pd.DataFrame({
                'model':model,'source':source,'seed':seed,'target':'BUSBRA',
                'unit_id':p.patient_key.astype(str),
                'y':p.y.astype(int),'score':p.score.astype(float)
            }))

lp_scores=pd.concat(lp_rows,ignore_index=True)
atomic_csv(lp_scores,OUT/'LINEAR_PROBE_TARGET_SCORES_RECOMPUTED.csv')

print("PASS linear-probe score reconstruction.")


## 10. External result tables

In [ ]:
scenarios=[
    ('BUSBRA','BREAST'),
    ('BUSBRA','BUSI'),
    ('BREAST','BUSBRA'),
    ('BREAST','BUSI'),
]

def metric_table(scores,method):
    rows=[]
    for model in MODELS:
        for source,target in scenarios:
            for seed in SEEDS:
                d=scores[
                    (scores.model==model)&
                    (scores.source==source)&
                    (scores.seed==seed)&
                    (scores.target==target)
                ]
                auc,ap=auc_ap(d.y,d.score)
                rows.append({
                    'method':method,'model':model,'source':source,'target':target,
                    'seed':seed,'n':len(d),'auc':auc,'auprc':ap
                })
    return pd.DataFrame(rows)

ft_metrics=metric_table(ft_scores,'FULL_FT')
lp_metrics=metric_table(lp_scores,'LINEAR_PROBE')
metrics=pd.concat([ft_metrics,lp_metrics],ignore_index=True)

atomic_csv(metrics,OUT/'RQ4_EXTERNAL_RESULTS.csv')
display(metrics.round(4))


## 11. Paired bootstrap engine

In [ ]:
def get_score(scores,model,source,target,seed):
    d=(
        scores[
            (scores.model==model)&
            (scores.source==source)&
            (scores.target==target)&
            (scores.seed==seed)
        ]
        .sort_values('unit_id')
        .reset_index(drop=True)
    )
    return (
        d.unit_id.astype(str).to_numpy(),
        d.y.to_numpy(int),
        d.score.to_numpy(float)
    )

def stratified_boot_indices(y,rng):
    y=np.asarray(y,dtype=int)
    i0=np.flatnonzero(y==0)
    i1=np.flatnonzero(y==1)
    return np.concatenate([
        rng.choice(i0,size=len(i0),replace=True),
        rng.choice(i1,size=len(i1),replace=True)
    ])

def paired_seed_balanced_boot(
    score_getter_a,
    score_getter_b,
    seed_base
):
    # getters(seed) -> ids,y,score
    ids,y,_=score_getter_a(SEEDS[0])

    for seed in SEEDS:
        ia,ya,_=score_getter_a(seed)
        ib,yb,_=score_getter_b(seed)
        assert np.array_equal(ids,ia)
        assert np.array_equal(ids,ib)
        assert np.array_equal(y,ya)
        assert np.array_equal(y,yb)

    seed_d=[]
    for seed in SEEDS:
        _,_,a=score_getter_a(seed)
        _,_,b=score_getter_b(seed)
        seed_d.append(roc_auc_score(y,a)-roc_auc_score(y,b))

    point=float(np.mean(seed_d))
    rng=np.random.default_rng(seed_base)
    boots=[]

    for _ in range(BOOTSTRAP_REPS):
        idx=stratified_boot_indices(y,rng)
        yy=y[idx]
        ds=[]
        for seed in SEEDS:
            _,_,a=score_getter_a(seed)
            _,_,b=score_getter_b(seed)
            ds.append(roc_auc_score(yy,a[idx])-roc_auc_score(yy,b[idx]))
        boots.append(float(np.mean(ds)))

    boots=np.asarray(boots,dtype=float)
    return {
        'point':point,
        'seed42_delta':float(seed_d[0]),
        'seed43_delta':float(seed_d[1]),
        'seed44_delta':float(seed_d[2]),
        'bootstrap_median':float(np.median(boots)),
        'ci_lo':float(np.quantile(boots,.025)),
        'ci_hi':float(np.quantile(boots,.975)),
        'p_a_better':float(np.mean(boots>0))
    },boots


## 12. A — Full-FT USFM vs MAE

In [ ]:
ft_adv_rows=[]
ft_adv_boot=[]
counter=0

for usfm in USFMS:
    for source,target in scenarios:
        ga=lambda seed,u=usfm,s=source,t=target: get_score(ft_scores,u,s,t,seed)
        gb=lambda seed,s=source,t=target: get_score(ft_scores,'MAE',s,t,seed)

        eff,b=paired_seed_balanced_boot(
            ga,gb,BOOTSTRAP_SEED+counter
        )
        counter+=1

        ft_adv_rows.append({
            'usfm':usfm,'source':source,'target':target,**eff
        })
        ft_adv_boot.append(pd.DataFrame({
            'usfm':usfm,'source':source,'target':target,
            'bootstrap_rep':np.arange(BOOTSTRAP_REPS),
            'ft_fm_minus_mae_auc':b
        }))

ft_adv=pd.DataFrame(ft_adv_rows)
ft_adv_boot=pd.concat(ft_adv_boot,ignore_index=True)

atomic_csv(ft_adv,OUT/'RQ4_FULLFT_USFM_VS_MAE_EFFECTS.csv')
atomic_csv(ft_adv_boot,OUT/'RQ4_FULLFT_USFM_VS_MAE_BOOTSTRAP.csv')
display(ft_adv.round(4))


## 13. B — Within-model Full-FT vs linear probe

In [ ]:
adapt_model_rows=[]
adapt_model_boot=[]

for model in MODELS:
    for source,target in scenarios:
        ga=lambda seed,m=model,s=source,t=target: get_score(ft_scores,m,s,t,seed)
        gb=lambda seed,m=model,s=source,t=target: get_score(lp_scores,m,s,t,seed)

        eff,b=paired_seed_balanced_boot(
            ga,gb,BOOTSTRAP_SEED+counter
        )
        counter+=1

        adapt_model_rows.append({
            'model':model,'source':source,'target':target,**eff
        })
        adapt_model_boot.append(pd.DataFrame({
            'model':model,'source':source,'target':target,
            'bootstrap_rep':np.arange(BOOTSTRAP_REPS),
            'fullft_minus_lp_auc':b
        }))

adapt_model=pd.DataFrame(adapt_model_rows)
adapt_model_boot=pd.concat(adapt_model_boot,ignore_index=True)

atomic_csv(adapt_model,OUT/'RQ4_FULLFT_VS_LINEAR_WITHIN_MODEL.csv')
atomic_csv(adapt_model_boot,OUT/'RQ4_FULLFT_VS_LINEAR_BOOTSTRAP.csv')
display(adapt_model.round(4))


## 14. C — Change in USFM-vs-MAE advantage after adaptation

For each bootstrap sample:

\[
[(FT_{FM}-FT_{MAE})-(LP_{FM}-LP_{MAE})].
\]


In [ ]:
change_rows=[]
change_boot=[]

for usfm in USFMS:
    for source,target in scenarios:
        # align target ids/y across all methods/models
        ids,y,_=get_score(ft_scores,usfm,source,target,SEEDS[0])

        for seed in SEEDS:
            for scores,model in [
                (ft_scores,usfm),(ft_scores,'MAE'),
                (lp_scores,usfm),(lp_scores,'MAE')
            ]:
                ii,yy,_=get_score(scores,model,source,target,seed)
                assert np.array_equal(ids,ii)
                assert np.array_equal(y,yy)

        point_seed=[]
        for seed in SEEDS:
            _,_,fu=get_score(ft_scores,usfm,source,target,seed)
            _,_,fm=get_score(ft_scores,'MAE',source,target,seed)
            _,_,lu=get_score(lp_scores,usfm,source,target,seed)
            _,_,lm=get_score(lp_scores,'MAE',source,target,seed)

            point_seed.append(
                (roc_auc_score(y,fu)-roc_auc_score(y,fm))
                -
                (roc_auc_score(y,lu)-roc_auc_score(y,lm))
            )

        point=float(np.mean(point_seed))
        rng=np.random.default_rng(BOOTSTRAP_SEED+counter)
        counter+=1
        boots=[]

        for _ in range(BOOTSTRAP_REPS):
            idx=stratified_boot_indices(y,rng)
            yy=y[idx]
            ds=[]

            for seed in SEEDS:
                _,_,fu=get_score(ft_scores,usfm,source,target,seed)
                _,_,fm=get_score(ft_scores,'MAE',source,target,seed)
                _,_,lu=get_score(lp_scores,usfm,source,target,seed)
                _,_,lm=get_score(lp_scores,'MAE',source,target,seed)

                ds.append(
                    (roc_auc_score(yy,fu[idx])-roc_auc_score(yy,fm[idx]))
                    -
                    (roc_auc_score(yy,lu[idx])-roc_auc_score(yy,lm[idx]))
                )

            boots.append(float(np.mean(ds)))

        boots=np.asarray(boots,dtype=float)

        change_rows.append({
            'usfm':usfm,'source':source,'target':target,
            'point':point,
            'seed42_change':float(point_seed[0]),
            'seed43_change':float(point_seed[1]),
            'seed44_change':float(point_seed[2]),
            'bootstrap_median':float(np.median(boots)),
            'ci_lo':float(np.quantile(boots,.025)),
            'ci_hi':float(np.quantile(boots,.975)),
            'p_change_positive':float(np.mean(boots>0))
        })

        change_boot.append(pd.DataFrame({
            'usfm':usfm,'source':source,'target':target,
            'bootstrap_rep':np.arange(BOOTSTRAP_REPS),
            'adaptation_change_in_usfm_advantage':boots
        }))

change=pd.DataFrame(change_rows)
change_boot=pd.concat(change_boot,ignore_index=True)

atomic_csv(change,OUT/'RQ4_ADAPTATION_CHANGE_IN_USFM_ADVANTAGE.csv')
atomic_csv(change_boot,OUT/'RQ4_ADAPTATION_CHANGE_BOOTSTRAP.csv')

display(change.round(4))


## 15. Frozen RQ4 interpretation and final experimental decision

In [ ]:
preservation_rows=[]
change_summary=[]

for usfm in USFMS:
    sub=ft_adv[ft_adv.usfm==usfm]
    pos=int((sub.point>0).sum())
    med=float(sub.point.median())
    supp=int((sub.p_a_better>=.90).sum())

    preserved=bool(
        pos>=3
        and med>=.02
        and supp>=3
    )

    preservation_rows.append({
        'usfm':usfm,
        'positive_ft_scenarios':pos,
        'median_ft_fm_minus_mae_auc':med,
        'bootstrap_supported_scenarios':supp,
        'advantage_preserved_after_fullft':preserved
    })

    cs=change[change.usfm==usfm]
    npos=int((cs.point>0).sum())
    nneg=int((cs.point<0).sum())
    cmed=float(cs.point.median())

    if npos>=3 and cmed>=.02:
        label='STRENGTHENED'
    elif nneg>=3 and cmed<=-.02:
        label='ATTENUATED'
    else:
        label='MIXED_OR_STABLE'

    change_summary.append({
        'usfm':usfm,
        'positive_adaptation_changes':npos,
        'negative_adaptation_changes':nneg,
        'median_change_in_fm_minus_mae_auc':cmed,
        'adaptation_effect_on_usfm_advantage':label
    })

preservation=pd.DataFrame(preservation_rows)
change_summary=pd.DataFrame(change_summary)

atomic_csv(
    preservation,
    OUT/'RQ4_FULLFT_ADVANTAGE_PRESERVATION_SUMMARY.csv'
)
atomic_csv(
    change_summary,
    OUT/'RQ4_ADAPTATION_EFFECT_SUMMARY.csv'
)

n_preserved=int(preservation.advantage_preserved_after_fullft.sum())

if n_preserved==2:
    claim_branch='BOTH_USFM_ADVANTAGES_PRESERVED_AFTER_FULLFT'
elif n_preserved==1:
    claim_branch='ONE_USFM_ADVANTAGE_PRESERVED_FULLFT_IS_MODEL_DEPENDENT'
else:
    claim_branch='BOTH_USFM_ADVANTAGES_NOT_PRESERVED_CLAIM_FROZEN_REPRESENTATIONS_ONLY'

decision={
    'experimental_phase':'COMPLETE',
    'next_step':'WRITE_MANUSCRIPT',
    'claim_branch':claim_branch,
    'preservation_summary':preservation.to_dict(orient='records'),
    'adaptation_change_summary':change_summary.to_dict(orient='records'),
    'no_rescue_rule':(
        'No additional USFM, PEFT method, readout, pooling rule, target tuning, dataset exclusion, '
        'or extra seed will be added after this result.'
    )
}
atomic_json(decision,OUT/'FINAL_EXPERIMENTAL_DECISION.json')

display(preservation.round(4))
display(change_summary.round(4))
print(json.dumps(decision,indent=2))


## 16. Package final-block results

In [ ]:
result_zip=ROOT/'FULLSTUDY_FINAL_FULLFT_RESULTS.zip'

files=[
    EXP/'FROZEN_FULLFT_CONFIG.json',
    OUT/'FULLFT_TRAINING_STATUS.csv',
    OUT/'FULLFT_ALL_HISTORIES.csv',
    OUT/'FULLFT_HISTORY_ARTIFACT_AUDIT.csv',
    OUT/'FULLFT_EXTERNAL_TARGET_SCORES.csv',
    OUT/'LINEAR_PROBE_TARGET_SCORES_RECOMPUTED.csv',
    OUT/'RQ4_EXTERNAL_RESULTS.csv',
    OUT/'RQ4_FULLFT_USFM_VS_MAE_EFFECTS.csv',
    OUT/'RQ4_FULLFT_USFM_VS_MAE_BOOTSTRAP.csv',
    OUT/'RQ4_FULLFT_VS_LINEAR_WITHIN_MODEL.csv',
    OUT/'RQ4_FULLFT_VS_LINEAR_BOOTSTRAP.csv',
    OUT/'RQ4_ADAPTATION_CHANGE_IN_USFM_ADVANTAGE.csv',
    OUT/'RQ4_ADAPTATION_CHANGE_BOOTSTRAP.csv',
    OUT/'RQ4_FULLFT_ADVANTAGE_PRESERVATION_SUMMARY.csv',
    OUT/'RQ4_ADAPTATION_EFFECT_SUMMARY.csv',
    OUT/'FINAL_EXPERIMENTAL_DECISION.json',
]

for p in files:
    if not p.exists():
        raise FileNotFoundError(p)

if result_zip.exists():
    result_zip.unlink()

with zipfile.ZipFile(result_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p,arcname=p.name)

with zipfile.ZipFile(result_zip,'r') as z:
    assert sorted(z.namelist())==sorted([p.name for p in files])

print("Created:",result_zip)
print("Size MB:",result_zip.stat().st_size/1e6)
print("FINAL RULE: after returning this ZIP, no new experiment is launched before manuscript review.")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

ROOT = Path('/content/drive/MyDrive/SPATIOSPECTRAL_PEFT_PILOT')
OUT = ROOT/'fullstudy_final_fullft_v1_0'/'outputs'

for name in [
    'RQ4_FULLFT_USFM_VS_MAE_EFFECTS.csv',
    'RQ4_FULLFT_VS_LINEAR_WITHIN_MODEL.csv',
    'RQ4_ADAPTATION_CHANGE_IN_USFM_ADVANTAGE.csv',
    'RQ4_ADAPTATION_CHANGE_BOOTSTRAP.csv',
]:
    print(name, '->', (OUT/name).exists())